In [ ]:
import subprocess, sys, os, glob

# Diagnose: show what's in /kaggle/input/
print('=== /kaggle/input/ contents ===')
for d in sorted(os.listdir('/kaggle/input')):
    print(f'  {d}')

# Search for onnxruntime wheel anywhere in /kaggle/input/
whl_candidates = glob.glob('/kaggle/input/**/*.whl', recursive=True)
print(f'\nFound wheels: {whl_candidates}')

# onnxruntime: try pre-installed first, then offline wheel
try:
    import onnxruntime as ort
    print(f'onnxruntime already available: {ort.__version__}')
except ImportError:
    if whl_candidates:
        ort_whl = next((w for w in whl_candidates if 'onnxruntime' in w), None)
        if ort_whl:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', ort_whl], check=True)
            print(f'onnxruntime installed from: {ort_whl}')
        else:
            raise RuntimeError(f'No onnxruntime wheel found. Wheels found: {whl_candidates}')
    else:
        raise RuntimeError('No wheels found in /kaggle/input/. Check dataset sources are attached.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'librosa', 'soundfile'], check=True)

import time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tv_models
import librosa
import soundfile as sf
import onnxruntime as ort
from collections import defaultdict
warnings.filterwarnings('ignore')

print(f'onnxruntime: {ort.__version__}')
print(f'torch: {torch.__version__}')

SR = 32000; DURATION = 5; N_FFT = 1024; HOP_LEN = 320
N_MELS = 128; FMIN = 20; FMAX = 16000; IMG_W = 160; INFER_BATCH = 32
PERCH_WEIGHT = 0.5; EFFNET_WEIGHT = 0.5

In [ ]:
COMP_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('train_audio', 'test_soundscapes', 'train_soundscapes')]
    if 'sample_submission.csv' in files:
        COMP_DIR = root
        break
assert COMP_DIR, 'Competition data not found'
print(f'COMP_DIR: {COMP_DIR}')

sub_df = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')
species_list = [c for c in sub_df.columns if c != 'row_id']
N_CLASSES = len(species_list)
print(f'Species: {N_CLASSES}')
print(f'Submission rows: {len(sub_df)}')

In [ ]:
# Search for PERCH ONNX model and labels anywhere in /kaggle/input/
print('=== Searching for PERCH files ===')
onnx_candidates  = glob.glob('/kaggle/input/**/*.onnx', recursive=True)
label_candidates = glob.glob('/kaggle/input/**/labels.csv', recursive=True)
print(f'ONNX files : {onnx_candidates}')
print(f'labels.csv : {label_candidates}')

perch_onnx_path = next((p for p in onnx_candidates if 'perch' in p.lower()), None)
if perch_onnx_path is None and onnx_candidates:
    perch_onnx_path = onnx_candidates[0]
assert perch_onnx_path, f'perch_v2.onnx not found. Found: {onnx_candidates}'
print(f'Using ONNX: {perch_onnx_path}')

perch_labels_path = next((p for p in label_candidates if 'perch' in p.lower()), None)
if perch_labels_path is None and label_candidates:
    perch_labels_path = label_candidates[0]
assert perch_labels_path, 'labels.csv not found'
print(f'Using labels: {perch_labels_path}')

so = ort.SessionOptions()
so.intra_op_num_threads = 4
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
perch_sess = ort.InferenceSession(perch_onnx_path, sess_options=so, providers=['CPUExecutionProvider'])

inp  = perch_sess.get_inputs()[0]
outs = perch_sess.get_outputs()
print(f'PERCH input : {inp.name}  shape={inp.shape}  type={inp.type}')
for o in outs:
    print(f'PERCH output: {o.name}  shape={o.shape}  type={o.type}')

# ── Species mapping ───────────────────────────────────────────────────
labels_df = pd.read_csv(perch_labels_path)
print(f'PERCH labels: {labels_df.shape}  columns: {labels_df.columns.tolist()}')
print(labels_df.head(3))

if 'bc_index' not in labels_df.columns:
    labels_df = labels_df.reset_index().rename(columns={'index': 'bc_index'})
sci_col = next(
    (c for c in labels_df.columns if c != 'bc_index' and
     any(k in c.lower() for k in ('inat', 'scientific', 'name', 'label', 'species'))),
    labels_df.columns[1]
)
labels_df = labels_df.rename(columns={sci_col: 'perch_label'})
print(f'Using column "{sci_col}" as PERCH label')

taxonomy = pd.read_csv(f'{COMP_DIR}/taxonomy.csv')
perch_label_to_idx = dict(zip(labels_df['perch_label'], labels_df['bc_index']))

mapping = taxonomy[['primary_label', 'scientific_name']].copy()
mapping['bc_index'] = mapping['scientific_name'].map(perch_label_to_idx).fillna(-1).astype(int)
n_mapped = (mapping['bc_index'] >= 0).sum()
print(f'Direct scientific name match: {n_mapped}/{N_CLASSES}')

label_to_bc = mapping.set_index('primary_label')['bc_index'].to_dict()
BC_INDICES = np.array([label_to_bc.get(s, -1) for s in species_list], dtype=np.int32)
print(f'BC_INDICES ready: {(BC_INDICES >= 0).sum()}/{N_CLASSES} species mapped')

In [ ]:
class BirdModel(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        base = tv_models.efficientnet_b0(weights=None)
        self.features = base.features
        self.pool     = base.avgpool
        n_feat        = base.classifier[1].in_features
        self.head     = nn.Sequential(nn.Dropout(0.3), nn.Linear(n_feat, n_classes))
    def forward(self, x):
        return self.head(torch.flatten(self.pool(self.features(x)), 1))

EFFNET_CANDIDATES = [
    # XC fold0
    '/kaggle/input/notebooks/gorubachohu/birdclef2026-efficientnet-training/model_fold0.pt',
    # SC fold0
    '/kaggle/input/notebooks/gorubachohu/birdclef2026-training-with-soundscapes-fold0/model_sc_fold0.pt',
    '/kaggle/input/datasets/gorubachohu/birdclef2026-sc-model-fold0/model_sc_fold0.pt',
    '/kaggle/input/birdclef2026-sc-model-fold0/model_sc_fold0.pt',
    # SC fold1 + fold2 (combined dataset)
    '/kaggle/input/birdclef2026-sc-models/model_sc_fold1.pt',
    '/kaggle/input/birdclef2026-sc-models/model_sc_fold2.pt',
    # SC fold1 (legacy individual dataset)
    '/kaggle/input/datasets/gorubachohu/birdclef2026-sc-model-fold1/model_sc_fold1.pt',
    '/kaggle/input/birdclef2026-sc-model-fold1/model_sc_fold1.pt',
]

effnet_models = []
loaded_names = set()
for path in EFFNET_CANDIDATES:
    name = os.path.basename(path)
    if name in loaded_names:
        continue
    if os.path.exists(path):
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        m = BirdModel(N_CLASSES)
        m.load_state_dict(ckpt['state'])
        m.eval()
        effnet_models.append(m)
        loaded_names.add(name)
        print(f'  Loaded {name} (val_auc={ckpt["auc"]:.4f})')
print(f'EfficientNet ensemble: {len(effnet_models)} models')

def audio_to_img(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LEN,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    mel = mel.astype(np.float32)
    if mel.shape[1] != IMG_W:
        mel = np.array([
            np.interp(np.linspace(0, mel.shape[1]-1, IMG_W),
                      np.arange(mel.shape[1]), mel[i])
            for i in range(mel.shape[0])], dtype=np.float32)
    return np.stack([mel, mel, mel], axis=0)

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -88, 88)))

soundscape_chunks = defaultdict(list)
for row_id in sub_df['row_id']:
    sc_name, end_sec = row_id.rsplit('_', 1)
    soundscape_chunks[sc_name].append(int(end_sec))
print(f'Soundscapes: {len(soundscape_chunks)}')

SOUNDSCAPE_DIR = f'{COMP_DIR}/test_soundscapes'
predictions = {}
perch_inp_name = perch_sess.get_inputs()[0].name
perch_out_name = perch_sess.get_outputs()[0].name

# Per-species weight: PERCH_WEIGHT only where mapped, else EfficientNet only
perch_mask = (BC_INDICES >= 0).astype(np.float32)   # 1=mapped, 0=unmapped
perch_w    = PERCH_WEIGHT * perch_mask               # 0.5 or 0.0
effnet_w   = 1.0 - perch_w                           # 0.5 or 1.0
n_mapped   = int(perch_mask.sum())
print(f'Blending: {n_mapped} species use PERCH+EfficientNet, {N_CLASSES-n_mapped} use EfficientNet only')

t_start = time.time()
for sc_idx, (sc_name, end_secs) in enumerate(soundscape_chunks.items()):
    sc_path = f'{SOUNDSCAPE_DIR}/{sc_name}.ogg'
    try:
        audio, _ = librosa.load(sc_path, sr=SR, mono=True)
    except Exception as e:
        print(f'[WARN] {sc_name}: {e}')
        audio = np.zeros(SR * 60, dtype=np.float32)

    n60 = 60 * SR
    audio = np.pad(audio, (0, max(0, n60 - len(audio))))[:n60]

    imgs, perch_chunks, row_ids = [], [], []
    for end_sec in sorted(end_secs):
        start_s = max(0, int((end_sec - DURATION) * SR))
        chunk = audio[start_s:start_s + DURATION * SR]
        if len(chunk) < DURATION * SR:
            chunk = np.pad(chunk, (0, DURATION * SR - len(chunk)))
        imgs.append(audio_to_img(chunk))
        perch_chunks.append(chunk.astype(np.float32))
        row_ids.append(f'{sc_name}_{end_sec}')

    imgs_arr  = np.stack(imgs)
    perch_arr = np.stack(perch_chunks)

    # ── EfficientNet inference ──────────────────────────────────
    effnet_probs = np.zeros((len(imgs_arr), N_CLASSES), dtype=np.float32)
    if effnet_models:
        for b0 in range(0, len(imgs_arr), INFER_BATCH):
            batch = torch.from_numpy(imgs_arr[b0:b0+INFER_BATCH])
            bp = np.zeros((len(batch), N_CLASSES), dtype=np.float32)
            for m in effnet_models:
                with torch.no_grad():
                    bp += torch.sigmoid(m(batch)).numpy()
            bp /= len(effnet_models)
            effnet_probs[b0:b0+len(batch)] = bp

    # ── PERCH inference ────────────────────────────────────────
    perch_probs = np.zeros((len(perch_chunks), N_CLASSES), dtype=np.float32)
    try:
        raw_logits = perch_sess.run([perch_out_name], {perch_inp_name: perch_arr})[0]
        raw_probs  = sigmoid(raw_logits)
        for i, bc_idx in enumerate(BC_INDICES):
            if 0 <= bc_idx < raw_probs.shape[1]:
                perch_probs[:, i] = raw_probs[:, bc_idx]
    except Exception as e:
        print(f'[WARN] PERCH inference failed for {sc_name}: {e}')

    # ── Adaptive blend: PERCH only where mapped ─────────────────
    blended = perch_w * perch_probs + effnet_w * effnet_probs

    for i, row_id in enumerate(row_ids):
        predictions[row_id] = blended[i]

    if (sc_idx + 1) % 10 == 0 or sc_idx == 0:
        elapsed = time.time() - t_start
        print(f'  [{sc_idx+1}/{len(soundscape_chunks)}] {elapsed:.0f}s')

print(f'Done in {time.time()-t_start:.0f}s')

In [ ]:
pred_matrix = np.zeros((len(sub_df), N_CLASSES), dtype=np.float32)
for i, row_id in enumerate(sub_df['row_id']):
    if row_id in predictions:
        pred_matrix[i] = predictions[row_id]

sub_df[species_list] = pred_matrix
sub_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved submission.csv  shape={sub_df.shape}')
print(f'Coverage: {sum(r in predictions for r in sub_df["row_id"])}/{len(sub_df)}')
print(sub_df.head(3))